# Feature Diagnostics

## Research Question

Our first objective of building and engineering Crypto Alpha Research Lab (CARL)
is to analyze the statistical distinction of momentum, volatility,
volume, and trend features

Specifically, we seek to answer the following questions:

1. What are the distributional characteristics of each feature?
2. How much data is lost because of rolling-window initialization?
3. Which features exhibit strong linear dependence?
4. Which feature pairs may contain redundant information?
5. What implications do these relationships have for subsequent
   predictive modeling and signal research?

## Why This Matters

Feature engineering does not automatically create useful information.

Multiple indicators may represent mathematically similar transformations
of the same underlying market variable. Including highly redundant
features in a predictive model can increase complexity without adding
meaningful information and may contribute to unstable parameter
estimates or overfitting.

Before testing predictive relationships, CARL therefore evaluates the
statistical structure of its engineered feature set.

The purpose of this analysis is diagnostic rather than prescriptive.
High correlation does not automatically imply that a feature should be
removed, because relationships between features may vary across market
regimes.

In [1]:

from crypto_alpha_lab.dataset import ResearchDataset

from crypto_alpha_lab.research.feature_matrix import (
    build_feature_matrix,
)

# import feature implementation from CARL
from crypto_alpha_lab.research.diagnostics import (
    feature_summary,
    feature_correlation,
    high_correlation_pairs,
    missing_feature_fraction,
)

In [2]:
# Load our dataset

btc = ResearchDataset.load(
    ticker="BTC-USD",
    start="2020-01-01",
    end="2026-01-01",
)

In [3]:
btc.prices.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-01,7200.174316,7254.330566,7174.944336,7194.892090,18565664997
2020-01-02,6985.470215,7212.155273,6935.270020,7202.551270,20802083465
2020-01-03,7344.884277,7413.715332,6914.996094,6984.428711,28111481032
2020-01-04,7410.656738,7427.385742,7309.514160,7345.375488,18444271275
2020-01-05,7411.317383,7544.497070,7400.535645,7410.451660,19725074095


In [4]:
# To build feature matrix

features = build_feature_matrix(
    btc,
    window=20,
    trend_long_window=60,
)

In [5]:
features.head()

,price_momentum,rolling_return,log_momentum,rolling_volatility,realized_volatility,relative_volume,volume_momentum,volume_zscore,price_to_moving_average,moving_average_spread
Date,,,,,,,,,,
2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
features.tail()

,price_momentum,rolling_return,log_momentum,rolling_volatility,realized_volatility,relative_volume,volume_momentum,volume_zscore,price_to_moving_average,moving_average_spread
Date,,,,,,,,,,
2020-12-26,0.366600,0.327291,0.312326,0.036763,0.035927,1.292223,0.910851,0.903385,0.234607,0.171915
2020-12-27,0.368945,0.328994,0.314041,0.036705,0.035870,1.688083,1.471706,2.014465,0.206959,0.177329
2020-12-28,0.478336,0.405278,0.390917,0.033793,0.032855,1.218797,0.547908,0.652845,0.219732,0.186434
2020-12-29,0.474753,0.402823,0.388490,0.033826,0.032886,1.109666,0.315092,0.332337,0.208270,0.195260
2020-12-30,0.579029,0.472430,0.456810,0.033538,0.032593,1.218825,1.007562,0.700040,0.244498,0.207178


In [7]:
features.shape

(365, 10)

In [8]:
summary = feature_summary(
    features
)

summary

,count,mean,std,min,25%,50%,75%,max
price_momentum,345.0,0.077329,0.171708,-0.486830,-0.014464,0.058804,0.209456,0.579029
rolling_return,345.0,0.076116,0.159374,-0.564882,-0.008453,0.062971,0.199688,0.472430
log_momentum,345.0,0.060460,0.173160,-0.667149,-0.014569,0.057140,0.190171,0.456810
rolling_volatility,345.0,0.032656,0.019481,0.011094,0.021297,0.030528,0.035564,0.109076
realized_volatility,345.0,0.033394,0.022941,0.011012,0.021082,0.030539,0.035179,0.124535
relative_volume,346.0,1.019746,0.263124,0.478557,0.830821,0.996335,1.187293,2.113337
volume_momentum,345.0,0.098628,0.419061,-0.655555,-0.213938,0.055333,0.339932,1.554959
volume_zscore,346.0,0.040841,1.129442,-1.985350,-0.863044,-0.016672,0.826813,3.756504
price_to_moving_average,346.0,0.032154,0.091719,-0.423231,-0.012358,0.029481,0.090352,0.244498
moving_average_spread,306.0,0.049423,0.117779,-0.274442,-0.017276,0.051750,0.145429,0.237630


In [9]:
# The percentage of NaN of each feature.
# Note that this is not a missing values from the original data
# 
missing = missing_feature_fraction(
    features
)

missing.sort_values(
    ascending=False
)

moving_average_spread      0.161644
price_momentum             0.054795
rolling_return             0.054795
log_momentum               0.054795
rolling_volatility         0.054795
realized_volatility        0.054795
volume_momentum            0.054795
relative_volume            0.052055
volume_zscore              0.052055
price_to_moving_average    0.052055
dtype: float64

In [10]:
correlation = feature_correlation(
    features
)

correlation.round(3)


,price_momentum,rolling_return,log_momentum,rolling_volatility,realized_volatility,relative_volume,volume_momentum,volume_zscore,price_to_moving_average,moving_average_spread
price_momentum,1.000,0.992,0.989,-0.344,-0.412,0.210,0.269,0.198,0.880,0.517
rolling_return,0.992,1.000,0.990,-0.303,-0.374,0.186,0.247,0.174,0.905,0.480
log_momentum,0.989,0.990,1.000,-0.432,-0.500,0.176,0.229,0.163,0.890,0.525
rolling_volatility,-0.344,-0.303,-0.432,1.000,0.995,-0.018,0.023,-0.015,-0.274,-0.421
realized_volatility,-0.412,-0.374,-0.500,0.995,1.000,-0.025,0.011,-0.017,-0.334,-0.446
relative_volume,0.210,0.186,0.176,-0.018,-0.025,1.000,0.752,0.956,0.200,0.058
volume_momentum,0.269,0.247,0.229,0.023,0.011,0.752,1.000,0.717,0.205,0.115
volume_zscore,0.198,0.174,0.163,-0.015,-0.017,0.956,0.717,1.000,0.177,0.044
price_to_moving_average,0.880,0.905,0.890,-0.274,-0.334,0.200,0.205,0.177,1.000,0.351
moving_average_spread,0.517,0.480,0.525,-0.421,-0.446,0.058,0.115,0.044,0.351,1.000


In [17]:
high_corr = high_correlation_pairs(
    features,
    threshold=0.80,
)

high_corr.sort_values(ascending=False, by='correlation')

,feature_1,feature_2,correlation
6,rolling_volatility,realized_volatility,0.995121
0,price_momentum,rolling_return,0.992347
3,rolling_return,log_momentum,0.989665
1,price_momentum,log_momentum,0.989002
7,relative_volume,volume_zscore,0.955989
4,rolling_return,price_to_moving_average,0.904507
5,log_momentum,price_to_moving_average,0.890188
2,price_momentum,price_to_moving_average,0.879682


## Think Like a Quant

From our results above, we can make the following observations:

The pairs of rolling_volatility and realized_volatility, price_momentum and rolling_return exhibit the strongest correlations of 0.9951 and 0.9923 respectively. Closely follow by correlations of rolling_return and log_momentum 0.9897, price_momentum and log_momentum 0.9890.This is because in crypto, price momentum and rolling returns are almost the same thing. 

price_momentum, rolling_return, and log_momentum are all derived from historical price changes. Their mathematical definitions differ slightly, but each measures recent price performance.
rolling_volatility and realized_volatility are both based on the dispersion of returns and therefore measure market risk.
relative_volume and volume_zscore both quantify unusual trading activity relative to recent history, although they use different normalization methods.

The high correlations therefore reflect both shared mathematical construction and common economic interpretation.

